In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2

In [ ]:
dataset_path = os.listdir('MangoLeafBD')
print(dataset_path)

In [ ]:
class_labels = []

for item in dataset_path:
 # Get all the file names
    all_classes = os.listdir('MangoLeafBD' + '/' +item)
 #print(all_classes)

 # Add them to the list
    for room in all_classes:
        class_labels.append((item, str('dataset_path' + '/' +item) + '/' + room))

In [ ]:
df = pd.DataFrame(data=class_labels, columns=['Labels', 'image'])
print(df.head())

In [ ]:
print(df.tail())

In [ ]:
print(df['Labels'].value_counts())

In [ ]:
path = 'MangoLeafBD/'

im_size = 224

images = []
labels = []

for i in dataset_path:
    data_path = path + str(i)
    filenames = [i for i in os.listdir(data_path)]
    
    for f in filenames:
        img = cv2.imread(data_path + '/' + f)
        img = cv2.resize(img, (im_size, im_size))
        images.append(img)
        labels.append(i)

In [ ]:
images = np.array(images)
images = images.astype('float32') / 255.0
images.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder , OneHotEncoder
import sys
np.set_printoptions(threshold=sys.maxsize)
y=df['Labels'].values
# print(y)
for i in range(0,len(y), 470):
    print(y[i])
y_labelencoder = LabelEncoder ()
y = y_labelencoder.fit_transform (y)
# print (y)
for i in range(0,len(y), 470):
    print(y[i])

In [ ]:
y=y.reshape(-1,1)

from sklearn.compose import ColumnTransformer
ct = ColumnTransformer([('my_ohe', OneHotEncoder(), [0])], remainder='passthrough')
Y = ct.fit_transform(y).toarray()


In [ ]:
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split


images, Y = shuffle(images, Y, random_state=1)


train_x, test_x, train_y, test_y = train_test_split(images, Y, test_size=0.2, random_state=415)

#inpect the shape of the training and testing.
print(train_x.shape)
print(train_y.shape)
print(test_x.shape)
print(test_y.shape)

In [ ]:
from matplotlib.pyplot import imread
from matplotlib.pyplot import imshow
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.imagenet_utils import decode_predictions
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG19

In [ ]:
NUM_CLASSES = 8
IMG_SIZE = 224

In [ ]:
def build_model(num_classes):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    #x = img_augmentation(inputs)
    x = inputs
    model = VGG19(include_top=False, input_tensor=x, weights="imagenet")

    # Freeze the pretrained weights
    model.trainable = False

    # Rebuild top
    x = layers.Flatten()(model.output)
#     x = layers.BatchNormalization()(x)

#     top_dropout_rate = 0.2
#     x = layers.Dropout(top_dropout_rate, name="top_dropout")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="pred")(x)

    # Compile
    model = tf.keras.Model(inputs, outputs, name="VGG19")
#     optimizer = tf.keras.optimizers.Adam(learning_rate=1e-2)
    model.compile(
        optimizer='adam', loss="categorical_crossentropy", metrics=["accuracy"]
    )
    return model

In [ ]:
model = build_model(num_classes=NUM_CLASSES)


In [ ]:
def plot_hist(hist):
    plt.plot(hist.history["accuracy"])
    #plt.plot(hist.history["val_accuracy"])
    plt.title("VGG19 Accuracy")
    plt.ylabel("Accuracy")
    plt.xlabel("Epochs")
    plt.legend(["train", "validation"], loc="upper left")
    plt.grid()
    plt.savefig('VGG19_acc',dpi=300)
    plt.show()

In [ ]:
epochs = 5


hist = model.fit(train_x, train_y, epochs=epochs, verbose=2)
    

In [ ]:
plot_hist(hist)

In [ ]:
preds = model.evaluate(test_x, test_y)
print ("Loss = " + str(preds[0]))
print ("Test Accuracy = " + str(preds[1]))

In [ ]:
import pickle

In [ ]:
with open('VGG19History','wb') as f:
    pickle.dump(hist.history,f)

In [ ]:
df = pd.DataFrame.from_dict(hist.history)
df.to_csv('VGG19_history.csv')

In [ ]:
model.save('VGG19_detect.h5')